# 02builddots: dots for the web page

PF7, "Kita: Malaysia in 3.2 million dots". Packs `data/processed/dots.parquet` into small binary files that the web page streams one after another, so the map fills in while it loads.

Each chunk file holds, in this order:

1. positions: longitude and latitude as 32-bit floats (8 bytes a dot)
2. attributes: group, sex, age band and urban (1 byte each, 4 bytes a dot)

The dots were shuffled in 01dots, so every chunk is an even sample of the whole country.

Output: `docs/data/dots_00.bin` ..., `docs/data/dots_index.json`, and whole-country images `docs/data/overview_*.png` for the zoomed-out view.

## 0. Settings

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()                                   # <- run from the PF7 folder
SRC = ROOT / "data" / "processed"
WEB = ROOT / "docs" / "data"
WEB.mkdir(parents=True, exist_ok=True)

N_CHUNKS = 8                                        # <- number of files the page loads in turn
GROUP_COLOURS = ["#3987e5", "#199e70", "#d95926", "#a6d854", "#5ec8e5", "#e8e2d0"]   # VISUAL: Malay, Other Bumiputera, Chinese, Indian, Other citizens, Non-citizens (the page reads them from dots_index.json)
OVERVIEW_M = 500                                    # <- pixel size of the whole-country images (metres)
OVERVIEW_CLIP = 99.5                                # VISUAL: percentile of busy pixels that counts as full brightness
OVERVIEW_GAMMA = 0.75                               # VISUAL: below 1 lifts faint countryside, above 1 darkens it

## 1. Read the dots

In [ ]:
dots = pd.read_parquet(SRC / "dots.parquet")
meta = json.loads((SRC / "dots_meta.json").read_text())
print(f"{len(dots):,} dots")
print(dots.head(3).to_string())

## 2. Write the chunks

In [ ]:
for old in WEB.glob("dots_*.bin"):
    old.unlink()
edges = np.linspace(0, len(dots), N_CHUNKS + 1).astype(int)
chunks = []
for i in range(N_CHUNKS):
    part = dots.iloc[edges[i]:edges[i + 1]]
    pos = np.column_stack([part["lon"].to_numpy(np.float32), part["lat"].to_numpy(np.float32)]).ravel()
    att = np.column_stack([part["group"], part["sex"], part["age"], part["urban"]]).astype(np.uint8).ravel()
    name = f"dots_{i:02d}.bin"
    with open(WEB / name, "wb") as f:
        f.write(pos.tobytes())
        f.write(att.tobytes())
    chunks.append({"file": name, "count": int(len(part))})
    print(f"{name}  {len(part):,} dots  {(WEB / name).stat().st_size / 1e6:.1f} MB")

## 3. Index file for the page

In [ ]:
counts = dots.groupby("group").size()
index = {}
index["n_dots"] = int(len(dots))
index["people_per_dot"] = meta["people_per_dot"]
index["census_year"] = meta["census_year"]
index["chunks"] = chunks
index["layout"] = "per chunk: float32 lon,lat pairs, then uint8 group,sex,age,urban quads"
index["groups"] = meta["groups"]
index["group_people"] = [int(counts.get(i, 0)) * meta["people_per_dot"] for i in range(len(meta["groups"]))]
index["ages"] = meta["ages"]
index["sex"] = meta["sex"]
index["urban"] = meta["urban"]
index["group_colours"] = GROUP_COLOURS
index["bounds"] = [float(dots["lon"].min()), float(dots["lat"].min()), float(dots["lon"].max()), float(dots["lat"].max())]
(WEB / "dots_index.json").write_text(json.dumps(index, indent=1))
print(json.dumps(index, indent=1)[:1200])

## 4. Read back one chunk (check)

In [ ]:
raw = (WEB / chunks[0]["file"]).read_bytes()
n = chunks[0]["count"]
pos = np.frombuffer(raw[:n * 8], dtype=np.float32).reshape(-1, 2)
att = np.frombuffer(raw[n * 8:], dtype=np.uint8).reshape(-1, 4)
first = dots.iloc[:n]
print("positions match:", np.allclose(pos[:, 0], first["lon"]), np.allclose(pos[:, 1], first["lat"]))
print("attributes match:", (att[:, 0] == first["group"].to_numpy()).all(), (att[:, 2] == first["age"].to_numpy()).all())

## 5. Whole-country images

At the whole-country view a screen pixel covers about 5 km and holds hundreds of dots, so live dots turn into one solid shape. Below zoom 7 the page shows these images instead, made from the same dots:

- `overview_all.png`: brightness from how many people live in each 500 m pixel (log scale)
- `overview_mix.png`: the same brightness, coloured by the mix of groups in the pixel
- `overview_g0.png` ... `overview_g5.png`: one group each (white, tinted by the page), on the same brightness scale

The images are in Web Mercator, the map's own projection, so their pixels line up with the map.

In [ ]:
from PIL import Image

R = 6378137.0
west, south, east, north = index["bounds"]
pad = 0.05
west, south, east, north = west - pad, south - pad, east + pad, north + pad


def merc_x(lon):
    return R * np.radians(lon)


def merc_y(lat):
    return R * np.log(np.tan(np.pi / 4 + np.radians(lat) / 2))


x0, x1 = merc_x(west), merc_x(east)
y0, y1 = merc_y(south), merc_y(north)
W = int(np.ceil((x1 - x0) / OVERVIEW_M))
H = int(np.ceil((y1 - y0) / OVERVIEW_M))
col = ((merc_x(dots["lon"].to_numpy(np.float64)) - x0) / (x1 - x0) * W).astype(np.int64).clip(0, W - 1)
row = ((y1 - merc_y(dots["lat"].to_numpy(np.float64))) / (y1 - y0) * H).astype(np.int64).clip(0, H - 1)
cell = row * W + col
grp = dots["group"].to_numpy()
counts = np.zeros((len(GROUP_COLOURS), H * W), dtype=np.float32)
for g in range(len(GROUP_COLOURS)):
    counts[g] = np.bincount(cell[grp == g], minlength=H * W)
total = counts.sum(axis=0)
print(f"image {W} x {H} pixels; busiest pixel {int(total.max()):,} dots")

In [ ]:
cmax = np.percentile(total[total > 0], OVERVIEW_CLIP)


def brightness(c):
    b = np.log1p(c) / np.log1p(cmax)
    return (np.clip(b, 0, 1) ** OVERVIEW_GAMMA * 255).astype(np.uint8).reshape(H, W)


def save_mask(alpha, name):
    img = np.dstack([np.full((H, W), 255, np.uint8), alpha])
    Image.fromarray(img, "LA").save(WEB / name, optimize=True)
    print(f"{name}  {(WEB / name).stat().st_size / 1e6:.2f} MB")


save_mask(brightness(total), "overview_all.png")
for g in range(len(GROUP_COLOURS)):
    save_mask(brightness(counts[g]), f"overview_g{g}.png")
rgb = np.array([[int(h[i:i + 2], 16) for i in (1, 3, 5)] for h in GROUP_COLOURS], dtype=np.float32)
share = counts / np.maximum(total, 1)
mix = (share.T @ rgb).reshape(H, W, 3)


def fill_empty(colour, has):
    # Give empty pixels the colour of their nearest neighbourhood, so the browser's smoothing does not darken the edges
    if has.all() or min(colour.shape[:2]) <= 1:
        return colour
    h, w = has.shape
    hp, wp = h + h % 2, w + w % 2
    c = np.zeros((hp, wp, 3), np.float32)
    m = np.zeros((hp, wp), np.float32)
    c[:h, :w] = colour * has[..., None]
    m[:h, :w] = has
    cs = c.reshape(hp // 2, 2, wp // 2, 2, 3).sum(axis=(1, 3))
    ms = m.reshape(hp // 2, 2, wp // 2, 2).sum(axis=(1, 3))
    coarse = fill_empty(cs / np.maximum(ms, 1e-6)[..., None], ms > 0)
    up = coarse.repeat(2, axis=0).repeat(2, axis=1)[:h, :w]
    return np.where(has[..., None], colour, up)


mix = fill_empty(mix, (total > 0).reshape(H, W)).clip(0, 255).astype(np.uint8)
Image.fromarray(np.dstack([mix, brightness(total)]), "RGBA").save(WEB / "overview_mix.png", optimize=True)
print(f"overview_mix.png  {(WEB / 'overview_mix.png').stat().st_size / 1e6:.2f} MB")
index["overview"] = {"bounds": [west, south, east, north], "width": W, "height": H, "pixel_m": OVERVIEW_M}
(WEB / "dots_index.json").write_text(json.dumps(index, indent=1))
print("index updated with the image bounds")

___

**END**